In [1]:
"""
Brain-to-Text: CTC (Connectionist Temporal Classification)
Predicts characters at each timestep, CTC handles alignment
No autoregressive generation = no mode collapse!
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import h5py
from tqdm.auto import tqdm
import string

# ============================================================================
# Configuration
# ============================================================================

class Config:
    # Paths
    DATA_ROOT = "/kaggle/input/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final"
    OUTPUT_DIR = "/kaggle/working"
    
    # Data
    NUM_FEATURES = 512
    
    # Model
    HIDDEN_DIM = 256
    NUM_LAYERS = 3
    DROPOUT = 0.3
    BIDIRECTIONAL = True
    
    # Training
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-3
    NUM_EPOCHS = 15
    VAL_SPLIT = 0.1
    
    # Sessions
    SESSIONS = [
        't15.2023.08.11', 't15.2023.08.13', 't15.2023.08.18', 't15.2023.08.20',
        't15.2023.08.25', 't15.2023.08.27', 't15.2023.09.01', 't15.2023.09.03',
        't15.2023.09.24', 't15.2023.09.29', 't15.2023.10.01', 't15.2023.10.06',
        't15.2023.10.08', 't15.2023.10.13', 't15.2023.10.15', 't15.2023.10.20',
        't15.2023.10.22', 't15.2023.11.03', 't15.2023.11.04', 't15.2023.11.17',
        't15.2023.11.19', 't15.2023.11.26', 't15.2023.12.03', 't15.2023.12.08',
        't15.2023.12.10', 't15.2023.12.17', 't15.2023.12.29', 't15.2024.02.25',
        't15.2024.03.03', 't15.2024.03.08', 't15.2024.03.15', 't15.2024.03.17',
        't15.2024.04.25', 't15.2024.04.28', 't15.2024.05.10', 't15.2024.06.14',
        't15.2024.07.19', 't15.2024.07.21', 't15.2024.07.28', 't15.2025.01.10',
        't15.2025.01.12', 't15.2025.03.14', 't15.2025.03.16', 't15.2025.03.30',
        't15.2025.04.13'
    ]
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 80)
print("BRAIN-TO-TEXT: CTC")
print("=" * 80)
print(f"Device: {Config.DEVICE}")
print(f"Architecture: BiLSTM + CTC Loss")
print(f"Prediction: Character-level (no autoregressive generation!)")
print("=" * 80)

# Clear GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ============================================================================
# Character Vocabulary
# ============================================================================

# Character set: a-z, space, apostrophe
CHARS = [' '] + list('abcdefghijklmnopqrstuvwxyz') + ["'"]
CHAR_TO_IDX = {char: idx for idx, char in enumerate(CHARS)}
IDX_TO_CHAR = {idx: char for idx, char in enumerate(CHARS)}
BLANK_IDX = len(CHARS)  # CTC blank token

print(f"\nCharacter vocabulary: {len(CHARS)} characters")
print(f"Characters: {CHARS[:10]}... (+ {len(CHARS)-10} more)")
print(f"Blank token index: {BLANK_IDX}")

# ============================================================================
# Dataset
# ============================================================================

class BrainDataset(Dataset):
    def __init__(self, session_files, is_train=True):
        self.session_files = session_files
        self.is_train = is_train
        self.samples = []
        
        for session, filepath in tqdm(session_files, desc="Indexing"):
            with h5py.File(filepath, 'r') as f:
                for key in f.keys():
                    self.samples.append((filepath, key))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        filepath, key = self.samples[idx]
        
        with h5py.File(filepath, 'r') as f:
            trial = f[key]
            neural = np.array(trial['input_features'], dtype=np.float32)
            neural_len = neural.shape[0]
            neural = torch.from_numpy(neural).float()
            
            if self.is_train:
                transcription = np.array(trial['transcription'], dtype=np.int32)
                text = ''.join([chr(int(c)) for c in transcription if c > 0])
                
                # Clean: lowercase, keep only valid chars
                text = text.lower()
                text = ''.join([c for c in text if c in CHAR_TO_IDX])
                
                # Convert to indices
                char_ids = [CHAR_TO_IDX[c] for c in text]
                char_ids = torch.tensor(char_ids, dtype=torch.long)
                text_len = len(char_ids)
                
                return neural, neural_len, char_ids, text_len, text
            else:
                return neural, neural_len

def collate_fn(batch):
    if len(batch[0]) == 5:  # Training
        neurals, neural_lens, char_ids_list, text_lens, texts = zip(*batch)
        
        # Pad neural sequences
        max_neural_len = max(neural_lens)
        neurals_padded = []
        for neural in neurals:
            if neural.size(0) < max_neural_len:
                pad = torch.zeros(max_neural_len - neural.size(0), Config.NUM_FEATURES)
                neural = torch.cat([neural, pad], dim=0)
            neurals_padded.append(neural)
        
        neurals_padded = torch.stack(neurals_padded)
        neural_lens = torch.tensor(neural_lens, dtype=torch.long)
        
        # Pad character sequences
        max_text_len = max(text_lens)
        char_ids_padded = []
        for char_ids in char_ids_list:
            if len(char_ids) < max_text_len:
                pad = torch.zeros(max_text_len - len(char_ids), dtype=torch.long)
                char_ids = torch.cat([char_ids, pad], dim=0)
            char_ids_padded.append(char_ids)
        
        char_ids_padded = torch.stack(char_ids_padded)
        text_lens = torch.tensor(text_lens, dtype=torch.long)
        
        return neurals_padded, neural_lens, char_ids_padded, text_lens, texts
    else:  # Test
        neurals, neural_lens = zip(*batch)
        max_neural_len = max(neural_lens)
        neurals_padded = []
        for neural in neurals:
            if neural.size(0) < max_neural_len:
                pad = torch.zeros(max_neural_len - neural.size(0), Config.NUM_FEATURES)
                neural = torch.cat([neural, pad], dim=0)
            neurals_padded.append(neural)
        
        neurals_padded = torch.stack(neurals_padded)
        neural_lens = torch.tensor(neural_lens, dtype=torch.long)
        return neurals_padded, neural_lens

# ============================================================================
# CTC Model
# ============================================================================

class BrainToTextCTC(nn.Module):
    """BiLSTM + CTC for character-level prediction."""
    
    def __init__(self, num_chars, hidden_dim, num_layers, dropout, bidirectional):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.num_chars = num_chars
        
        # Input projection
        self.input_proj = nn.Linear(Config.NUM_FEATURES, hidden_dim)
        
        # BiLSTM encoder
        self.lstm = nn.LSTM(
            hidden_dim, hidden_dim, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        # Output projection
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.output_proj = nn.Linear(lstm_output_dim, num_chars + 1)  # +1 for blank
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, neural, lengths):
        """
        neural: (batch, time, 512)
        lengths: (batch,) actual lengths
        Returns: (batch, time, num_chars+1) log probabilities
        """
        # Project input
        x = self.input_proj(neural)
        
        # Pack for variable lengths
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        # LSTM
        packed_output, _ = self.lstm(packed)
        
        # Unpack
        output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        
        # Project to character space
        output = self.dropout(output)
        logits = self.output_proj(output)  # (batch, time, num_chars+1)
        
        # Log softmax for CTC
        log_probs = F.log_softmax(logits, dim=-1)
        
        return log_probs

# ============================================================================
# CTC Greedy Decoder
# ============================================================================

def ctc_greedy_decode(log_probs, lengths):
    """
    Greedy CTC decoding.
    log_probs: (batch, time, num_chars+1)
    lengths: (batch,)
    Returns: list of decoded strings
    """
    batch_size = log_probs.size(0)
    predictions = []
    
    for i in range(batch_size):
        # Get most likely character at each timestep
        seq_len = lengths[i]
        pred_ids = log_probs[i, :seq_len].argmax(dim=-1).cpu().numpy()
        
        # Remove consecutive duplicates and blanks
        chars = []
        prev_idx = None
        for idx in pred_ids:
            if idx != BLANK_IDX and idx != prev_idx:
                chars.append(IDX_TO_CHAR[idx])
            prev_idx = idx
        
        text = ''.join(chars)
        predictions.append(text)
    
    return predictions

# ============================================================================
# Training
# ============================================================================

def train_model():
    print("\n" + "=" * 80)
    print("TRAINING")
    print("=" * 80)
    
    # Load data
    train_session_files = []
    for session in Config.SESSIONS:
        train_file = Path(Config.DATA_ROOT) / session / "data_train.hdf5"
        if train_file.exists():
            train_session_files.append((session, str(train_file)))
    
    val_count = max(1, int(len(train_session_files) * Config.VAL_SPLIT))
    val_files = train_session_files[:val_count]
    train_files = train_session_files[val_count:]
    
    train_dataset = BrainDataset(train_files, is_train=True)
    val_dataset = BrainDataset(val_files, is_train=True)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, 
                             shuffle=True, collate_fn=collate_fn, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, 
                           shuffle=False, collate_fn=collate_fn, num_workers=2)
    
    # Model
    model = BrainToTextCTC(
        len(CHARS), Config.HIDDEN_DIM, Config.NUM_LAYERS, 
        Config.DROPOUT, Config.BIDIRECTIONAL
    ).to(Config.DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")
    print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")
    
    # CTC Loss
    ctc_loss = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    
    for epoch in range(Config.NUM_EPOCHS):
        # Train
        model.train()
        train_loss = 0
        
        print(f"\n{'='*80}")
        print(f"EPOCH {epoch+1}/{Config.NUM_EPOCHS} - TRAINING")
        print('='*80)
        
        for batch_idx, (neural, neural_lens, char_ids, text_lens, _) in enumerate(tqdm(train_loader, desc="Training")):
            neural = neural.to(Config.DEVICE)
            neural_lens = neural_lens.to(Config.DEVICE)
            char_ids = char_ids.to(Config.DEVICE)
            text_lens = text_lens.to(Config.DEVICE)
            
            # Forward
            log_probs = model(neural, neural_lens)  # (batch, time, num_chars+1)
            
            # CTC loss expects: (time, batch, num_chars+1)
            log_probs = log_probs.permute(1, 0, 2)
            
            # CTC loss
            loss = ctc_loss(log_probs, char_ids, neural_lens, text_lens)
            
            if torch.isnan(loss):
                print(f"WARNING: NaN loss at batch {batch_idx}, skipping")
                continue
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            
            if batch_idx % 50 == 0:
                print(f"  Batch {batch_idx}, Loss: {loss.item():.4f}")
        
        avg_train_loss = train_loss / len(train_loader)
        print(f"\nTraining Loss: {avg_train_loss:.4f}")
        
        # Validate
        print(f"\n{'='*80}")
        print(f"EPOCH {epoch+1}/{Config.NUM_EPOCHS} - VALIDATION")
        print('='*80)
        
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for batch_idx, (neural, neural_lens, char_ids, text_lens, true_texts) in enumerate(val_loader):
                neural = neural.to(Config.DEVICE)
                neural_lens = neural_lens.to(Config.DEVICE)
                char_ids = char_ids.to(Config.DEVICE)
                text_lens = text_lens.to(Config.DEVICE)
                
                log_probs = model(neural, neural_lens)
                log_probs_ctc = log_probs.permute(1, 0, 2)
                loss = ctc_loss(log_probs_ctc, char_ids, neural_lens, text_lens)
                val_loss += loss.item()
                
                if batch_idx == 0:
                    # Decode predictions
                    predictions = ctc_greedy_decode(log_probs[:5], neural_lens[:5])
                    
                    for i in range(min(5, len(predictions))):
                        print(f"{i+1}. Pred: \"{predictions[i]}\"")
                        print(f"   True: \"{true_texts[i]}\"")
        
        avg_val_loss = val_loss / len(val_loader)
        print(f"\nValidation Loss: {avg_val_loss:.4f}")
        
        scheduler.step(avg_val_loss)
        print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save
    torch.save(model.state_dict(), Path(Config.OUTPUT_DIR) / "ctc_model.pt")
    print(f"\n[OK] Model saved")
    
    return model

# ============================================================================
# Generate Submission
# ============================================================================

def generate_submission(model):
    print("\n" + "=" * 80)
    print("GENERATING SUBMISSION")
    print("=" * 80)
    
    test_files = []
    for session in Config.SESSIONS:
        test_file = Path(Config.DATA_ROOT) / session / "data_test.hdf5"
        if test_file.exists():
            test_files.append((session, str(test_file)))
    
    test_dataset = BrainDataset(test_files, is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, 
                            shuffle=False, collate_fn=collate_fn, num_workers=2)
    
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for neural, neural_lens in tqdm(test_loader, desc="Predicting"):
            neural = neural.to(Config.DEVICE)
            neural_lens = neural_lens.to(Config.DEVICE)
            
            log_probs = model(neural, neural_lens)
            batch_preds = ctc_greedy_decode(log_probs, neural_lens)
            
            predictions.extend(batch_preds)
            
            if len(predictions) <= 10:
                for i, text in enumerate(batch_preds[:10 - len(predictions) + len(batch_preds)]):
                    print(f"  {len(predictions) - len(batch_preds) + i + 1}. \"{text}\"")
    
    submission = pd.DataFrame({'id': range(len(predictions)), 'text': predictions})
    submission.to_csv(Path(Config.OUTPUT_DIR) / "submission.csv", index=False)
    print(f"\n[OK] Saved submission.csv ({len(predictions)} rows)")

# ============================================================================
# Main
# ============================================================================

if __name__ == "__main__":
    model = train_model()
    generate_submission(model)
    print("\n" + "=" * 80)
    print("COMPLETE!")
    print("=" * 80)

BRAIN-TO-TEXT: CTC
Device: cuda
Architecture: BiLSTM + CTC Loss
Prediction: Character-level (no autoregressive generation!)

Character vocabulary: 28 characters
Characters: [' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i']... (+ 18 more)
Blank token index: 28

TRAINING


Indexing:   0%|          | 0/41 [00:00<?, ?it/s]

Indexing:   0%|          | 0/4 [00:00<?, ?it/s]

Model parameters: 4,352,797
Train: 6,961 | Val: 1,111

EPOCH 1/15 - TRAINING


Training:   0%|          | 0/436 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b008a554cc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b008a554cc0>^^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
^    ^self._shutdown_workers()^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
^    ^if w.is_alive():^
^ ^ ^ ^^ ^ ^ 


Validation Loss: 1.3974
LR: 0.000500

[OK] Model saved

GENERATING SUBMISSION


Indexing:   0%|          | 0/41 [00:00<?, ?it/s]

Predicting:   0%|          | 0/91 [00:00<?, ?it/s]


[OK] Saved submission.csv (1450 rows)

COMPLETE!
